In [ ]:
from llmcompressor import oneshot
from llmcompressor.modifiers.pruning.sparsegpt import SparseGPTModifier

modifier_kwargs = {
    "targets": ["Linear"],
    "ignore": ["lm_head"],
    "sparsity": 0.2,
}
#modifier_kwargs["mask_structure"] = "2:4"

recipe = [SparseGPTModifier(**modifier_kwargs)]

In [3]:
from huggingface_hub import login

login()

In [ ]:
import torch
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
model_path = "meta-llama/Llama-2-7b-chat-hf"
device = "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype=torch.bfloat16,
    device_map=device,
    )

In [5]:
from datasets import load_dataset, Dataset

traindata = load_dataset('allenai/c4', data_files={'train': 'en/c4-train.00000-of-01024.json.gz'}, split='train')

In [ ]:
oneshot(
    model=model,
    tokenizer=tokenizer,
    dataset=traindata, #calib_data,
    recipe=recipe,
    max_seq_length=2048,
    num_calibration_samples=1024,
)